# M4 · Conteo de personas por segmentación de instancias — evaluación (Bono +5)
**IAM-811 · Laboratorio Final — Sistema Inteligente de Control de Peaje**

| | |
|---|---|
| **Modelo** | `yolo11s-seg` **preentrenado en COCO** (clase 0 = `person`), **sin fine-tuning** |
| **Dataset de evaluación** | [*Car and Persons*](https://universe.roboflow.com/yolov8-carsperson/car-and-persons) — YOLOv8 CarsPerson, Roboflow Universe, CC BY 4.0, v1 · `yolov11` |
| **Clase usada** | solo `Person` (por nombre, sin distinguir mayúsculas) |
| **Métrica** | error de conteo por imagen: **MAE, sesgo, % exacto, % dentro de ±1** (splits valid y test) |
| **Salida** | `m4_pasajeros_seg.pt` = copia de los pesos COCO (carga en `inferencia/app.py` sin cambios) |

El fine-tuning queda detrás de `HACER_FINETUNE = False` (sección 6).

> ### 🔒 Restricción de privacidad (obligatoria)
> * **Prohibido** el reconocimiento facial, la identificación de personas o el cálculo de *embeddings* de rostro.
> * **Nunca** se guardan recortes de personas ni de rostros (ni en disco, ni en Drive, ni en la BD).
> * Las visualizaciones pintan cada persona con una **máscara opaca** y muestran solo el **número de personas**.

> ### ⚠️ Limitación declarada
> *Car and Persons* contiene **peatones y autos en la calle**, no pasajeros fotografiados **dentro** de un
> vehículo a través del parabrisas. Este notebook mide qué tan bien cuenta personas el modelo COCO en general;
> no mide el caso exacto de la caseta (personas parcialmente visibles tras el vidrio, reflejos). En producción el
> servidor además exige que la persona caiga **dentro de la máscara del vehículo** (M2) para contarla como pasajero.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "Sin GPU: la evaluación también corre en CPU (más lento)"
!pip -q install ultralytics==8.4.218 roboflow
import os, sys, json, glob, time, shutil, subprocess, yaml, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

## 2. Configuración (editable)

In [ ]:
RF_WORKSPACE, RF_PROYECTO, RF_VERSION, RF_FORMATO = 'yolov8-carsperson', 'car-and-persons', 1, 'yolov11'
DATASET_INFO = {'titulo': 'Car and Persons', 'autor': 'YOLOv8 CarsPerson', 'licencia': 'CC BY 4.0',
                'url': 'https://universe.roboflow.com/yolov8-carsperson/car-and-persons',
                'workspace': RF_WORKSPACE, 'proyecto': RF_PROYECTO, 'version': RF_VERSION, 'formato': RF_FORMATO}
CLASE_PERSONA = 'person'
CONF = 0.25                      # umbral de confianza para contar una persona
SPLITS_EVAL = ['valid', 'test']
HACER_FINETUNE = False           # True: convierte cajas a máscaras con SAM y ajusta yolo11s-seg (sección 6)
EPOCHS_FT, TOPE_HORAS_FT = 30, 0.5
REPO_URL = 'https://github.com/jayutomas623/sistema-peaje-ia'
DIR_DRIVE = '/content/drive/MyDrive/peaje-ia'
M4_PT = '/content/m4_pasajeros_seg.pt'

In [ ]:
# === UTILIDADES COLAB (API key, Drive, servidor de inferencia del repo) ===
def obtener_api_key():
    # 1) Secreto de Colab 'ROBOFLOW_API_KEY' (🔑 en la barra lateral, con acceso habilitado para este notebook)
    # 2) Si no existe, se pide con getpass (no queda escrita en el notebook ni en el repositorio)
    try:
        from google.colab import userdata
        clave = userdata.get('ROBOFLOW_API_KEY')
        if clave:
            return clave
    except Exception as e:
        print('Secreto ROBOFLOW_API_KEY no disponible (' + type(e).__name__ + '): se pedirá a mano.')
    from getpass import getpass
    return getpass('Pegue su ROBOFLOW_API_KEY (no se guarda): ').strip()

def numero_version(v):
    # Objeto Version del SDK de Roboflow -> número entero (tolera distintas versiones del SDK)
    for valor in (getattr(v, 'version', None), getattr(v, 'id', None), v):
        try:
            return int(str(valor).rstrip('/').split('/')[-1])
        except (TypeError, ValueError):
            continue
    return -1

def descargar_roboflow(workspace, proyecto, version, formato):
    from roboflow import Roboflow
    rf = Roboflow(api_key=obtener_api_key())
    p = rf.workspace(workspace).project(proyecto)
    try:
        versiones = sorted(numero_version(v) for v in p.versions())
    except Exception as e:
        versiones = []; print('No se pudo listar versiones:', e)
    print(f'{workspace}/{proyecto} · versiones disponibles: {versiones} · se usa v{version}')
    if versiones and max(versiones) != version:
        print(f'⚠️  La última versión publicada es v{max(versiones)}. Se mantiene v{version} (la documentada en '
              f'entrenamiento/DATASETS.md); si cambia RF_VERSION, actualice también DATASETS.md.')
    return p.version(version).download(formato).location

def clonar_repo(destino='/content/sistema-peaje-ia'):
    if not os.path.isdir(destino):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, destino], check=True)
    return destino

def importar_servidor(repo_dir, **env):
    # Importa inferencia/peaje_ia (el MISMO código que corre en RunPod) con las variables de entorno dadas
    for k, v in env.items():
        os.environ[k] = str(v)
    for m in [m for m in sys.modules if m == 'peaje_ia' or m.startswith('peaje_ia.')]:
        del sys.modules[m]                      # recarga config.py con las nuevas rutas de pesos
    ruta = os.path.join(repo_dir, 'inferencia')
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

def guardar_en_drive(archivos, subcarpeta=''):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive no montado:', e); return
    for a in archivos:
        if not os.path.exists(a):
            print('  (no existe, se omite)', a); continue
        # pesos -> MyDrive/peaje-ia/pesos/ · métricas, CSV y gráficos -> MyDrive/peaje-ia/<subcarpeta>/
        dest = os.path.join(DIR_DRIVE, 'pesos' if a.endswith('.pt') else subcarpeta)
        os.makedirs(dest, exist_ok=True)
        shutil.copy(a, dest); print('  ->', os.path.join(dest, os.path.basename(a)))

def tope_de_tiempo(horas):
    # Tope de seguridad para Colab: detiene el entrenamiento al terminar la época en que se supera `horas`.
    # (No se usa el argumento `time=` de Ultralytics porque recalcula `epochs` para llenar el tiempo.)
    inicio = time.time()
    def cb(trainer):
        if time.time() - inicio > horas * 3600:
            print(f'⏱️  Tope de {horas} h alcanzado: se detiene tras la época {trainer.epoch + 1}.')
            trainer.stop = True
    return cb

In [ ]:
# === FUNCIONES M4: GT de conteo y métricas (probadas localmente con datos sintéticos) ===
import os, glob

def normalizar_nombre(n):
    return str(n).strip().lower().replace('_', '-').replace(' ', '-')

def nombres_de_yaml(cfg):
    n = cfg['names']
    return list(n) if isinstance(n, (list, tuple)) else [n[k] for k in sorted(n, key=int)]

def ids_clase(nombres, objetivo):
    # todos los ids cuyo nombre coincide sin distinguir mayúsculas ('Person', 'person', ...)
    ids = {i for i, n in enumerate(nombres) if normalizar_nombre(n) == normalizar_nombre(objetivo)}
    if not ids:
        raise ValueError(f'No existe la clase "{objetivo}" en data.yaml. names = {nombres}')
    return ids

def contar_gt(ruta_lbl, ids):
    # GT de conteo = número de cajas (o polígonos) de la clase persona en la imagen
    if not os.path.exists(ruta_lbl):
        return 0
    with open(ruta_lbl, encoding='utf-8') as fh:
        return sum(1 for l in fh.read().splitlines() if l.split() and int(float(l.split()[0])) in ids)

def metricas_conteo(gt, pred):
    assert len(gt) == len(pred) and gt, 'listas vacías o de distinto tamaño'
    err = [p - g for g, p in zip(gt, pred)]
    n = len(err)
    return {'n': n, 'MAE': sum(abs(e) for e in err) / n, 'sesgo': sum(err) / n,
            'exacto_pct': 100 * sum(e == 0 for e in err) / n, 'dentro_1_pct': 100 * sum(abs(e) <= 1 for e in err) / n}

## 3. Descarga del dataset de evaluación

In [ ]:
DS_DIR = descargar_roboflow(RF_WORKSPACE, RF_PROYECTO, RF_VERSION, RF_FORMATO)
print(open(f'{DS_DIR}/data.yaml').read())
NOMBRES = nombres_de_yaml(yaml.safe_load(open(f'{DS_DIR}/data.yaml')))
IDS_PERSONA = ids_clase(NOMBRES, CLASE_PERSONA)
SPLITS = [s for s in SPLITS_EVAL if os.path.isdir(f'{DS_DIR}/{s}/images')]
print(f'names: {NOMBRES} · ids persona: {sorted(IDS_PERSONA)} · splits evaluados: {SPLITS}')

## 4. Evaluación del conteo con YOLO11s-seg COCO (sin entrenamiento)

In [ ]:
coco = YOLO('yolo11s-seg.pt')
assert coco.names[0] == 'person', coco.names[0]

def evaluar(modelo, clase_pred=0):
    filas = []
    for s in SPLITS:
        for ruta in sorted(glob.glob(f'{DS_DIR}/{s}/images/*')):
            base = os.path.splitext(os.path.basename(ruta))[0]
            gt = contar_gt(f'{DS_DIR}/{s}/labels/{base}.txt', IDS_PERSONA)
            r = modelo.predict(ruta, conf=CONF, classes=[clase_pred], device=DEVICE, verbose=False)[0]
            filas.append({'split': s, 'imagen': os.path.basename(ruta), 'personas_gt': gt,
                          'personas_pred': int(len(r.boxes)), 'error': int(len(r.boxes)) - gt})
    df = pd.DataFrame(filas)
    res = {'global': metricas_conteo(df.personas_gt.tolist(), df.personas_pred.tolist())}
    for s, g in df.groupby('split'):
        res[s] = metricas_conteo(g.personas_gt.tolist(), g.personas_pred.tolist())
    return df, res

t0 = time.time()
df_coco, MET_COCO = evaluar(coco)
TIEMPO_EVAL_MIN = round((time.time() - t0) / 60, 1)
df_coco.to_csv('/content/m4_eval_conteo.csv', index=False)
display(pd.DataFrame(MET_COCO).T.round(3))
display(pd.crosstab(df_coco.personas_gt.clip(upper=8), df_coco.personas_pred.clip(upper=8)))

## 5. Visualización con privacidad (máscara opaca + conteo, sin rostros)

In [ ]:
def dibujar_privado(img, r):
    # Cada persona se cubre con una máscara OPACA: el rostro queda oculto. Solo se muestra el conteo.
    out = img.copy()
    if r.masks is not None:
        H, W = out.shape[:2]
        for poly in r.masks.xy:
            if len(poly) >= 3:
                m = np.zeros((H, W), np.uint8); cv2.fillPoly(m, [poly.astype(np.int32)], 1)
                out[m.astype(bool)] = (90, 60, 30)
    cv2.putText(out, f'Personas: {len(r.boxes)}', (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)
    return out

for _, f in df_coco[df_coco.personas_gt > 0].head(4).iterrows():
    img = cv2.imread(f'{DS_DIR}/{f.split}/images/{f.imagen}')
    r = coco.predict(img, conf=CONF, classes=[0], device=DEVICE, verbose=False)[0]
    plt.figure(figsize=(9, 6)); plt.imshow(dibujar_privado(img, r)[:, :, ::-1]); plt.axis('off')
    plt.title(f'GT: {f.personas_gt} · predicho: {f.personas_pred}'); plt.show()
# Nota: no se escribe a disco ninguna imagen ni recorte de personas.

## 6. Fine-tuning opcional (`HACER_FINETUNE = True`)
El dataset solo trae cajas. Para entrenar segmentación se convierten las cajas `Person` en máscaras con
**SAM** (`ultralytics.data.converter.yolo_bbox2segment`) y se ajusta `yolo11s-seg` con una sola clase
`person`. Por defecto **no se ejecuta**: el modelo entregado es el COCO preentrenado.

In [ ]:
MODELO_ENTREGADO, MET_FT = 'coco', None
if HACER_FINETUNE:
    from ultralytics.data.converter import yolo_bbox2segment
    FT_DIR = '/content/m4_ft'; shutil.rmtree(FT_DIR, ignore_errors=True)
    splits_ft = [s for s in ('train', 'valid', 'test') if os.path.isdir(f'{DS_DIR}/{s}/images')]
    for s in splits_ft:
        os.makedirs(f'{FT_DIR}/{s}/labels_det'); os.symlink(f'{DS_DIR}/{s}/images', f'{FT_DIR}/{s}/images')
        for f in glob.glob(f'{DS_DIR}/{s}/labels/*.txt'):
            lineas = [l.split() for l in open(f).read().splitlines() if l.split()]
            persona = ['0 ' + ' '.join(t[1:5]) for t in lineas if int(float(t[0])) in IDS_PERSONA and len(t) == 5]
            open(f'{FT_DIR}/{s}/labels_det/{os.path.basename(f)}', 'w').write('\n'.join(persona))
        os.rename(f'{FT_DIR}/{s}/labels_det', f'{FT_DIR}/{s}/labels')
        yolo_bbox2segment(f'{FT_DIR}/{s}/images', save_dir=f'{FT_DIR}/{s}/labels_seg', sam_model='sam_b.pt')
        shutil.rmtree(f'{FT_DIR}/{s}/labels'); os.rename(f'{FT_DIR}/{s}/labels_seg', f'{FT_DIR}/{s}/labels')
    yaml.safe_dump({'path': FT_DIR, 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'person'}},
                   open(f'{FT_DIR}/data.yaml', 'w'))
    ft = YOLO('yolo11s-seg.pt'); ft.add_callback('on_fit_epoch_end', tope_de_tiempo(TOPE_HORAS_FT))
    ft.train(data=f'{FT_DIR}/data.yaml', epochs=EPOCHS_FT, imgsz=640, batch=16, patience=10,
             project='/content/runs/segment', name='m4', exist_ok=True, device=DEVICE)
    best_ft = YOLO(f'{ft.trainer.save_dir}/weights/best.pt')
    df_ft, MET_FT = evaluar(best_ft)
    display(pd.DataFrame({'COCO': MET_COCO['global'], 'fine-tuning': MET_FT['global']}).T.round(3))
    if MET_FT['global']['MAE'] < MET_COCO['global']['MAE']:
        MODELO_ENTREGADO = f'{ft.trainer.save_dir}/weights/best.pt'
        df_ft.to_csv('/content/m4_eval_conteo.csv', index=False)
print('Modelo entregado:', MODELO_ENTREGADO)

## 7. Exportar `m4_pasajeros_seg.pt` y comprobar que carga en el servidor
Se copia el modelo entregado y se carga con `ContadorPasajeros` de `inferencia/peaje_ia/m4_pasajeros.py`.

> Con los pesos COCO, `/health` del servidor mostrará M4 en modo `entrenado` porque existe el archivo, aunque
> son los mismos pesos que su modo *fallback-coco*. El campo `modelo` de `m4_metricas.json` lo deja explícito.

In [ ]:
PESOS_COCO = getattr(coco, 'ckpt_path', None) or 'yolo11s-seg.pt'
shutil.copy(PESOS_COCO if MODELO_ENTREGADO == 'coco' else MODELO_ENTREGADO, M4_PT)
REPO_DIR = clonar_repo()
importar_servidor(REPO_DIR, M4_PESOS=M4_PT, DEVICE='auto')
from peaje_ia.m4_pasajeros import ContadorPasajeros, NOMBRES_PERSONA
cp = ContadorPasajeros()
assert cp.modo == 'entrenado', cp.nombre
img = cv2.imread(f"{DS_DIR}/{df_coco.iloc[0].split}/images/{df_coco.iloc[0].imagen}")
nombres_detectados = {d['nombre'].lower() for d in cp.predecir_crudo(img)}
assert 'person' in {n.lower() for n in cp.modelo.names.values()} and 'person' in NOMBRES_PERSONA
print('Servidor:', cp.nombre, '· clases detectadas en una imagen de prueba:', nombres_detectados)

## 8. `m4_metricas.json` y copia en Google Drive

In [ ]:
metricas_json = {
    'modelo': 'yolo11s-seg preentrenado COCO, sin fine-tuning' if MODELO_ENTREGADO == 'coco'
              else 'yolo11s-seg fine-tuning (máscaras SAM desde cajas Person)',
    'archivo': 'm4_pasajeros_seg.pt', 'dataset': DATASET_INFO, 'clase_gt': CLASE_PERSONA,
    'ids_persona_en_dataset': sorted(IDS_PERSONA), 'conf': CONF, 'splits': SPLITS,
    'gt': 'número de cajas Person por imagen', 'prediccion': f'personas COCO (clase 0) con conf >= {CONF}',
    'conteo': MET_COCO, 'conteo_fine_tuning': MET_FT, 'tiempo_eval_min': TIEMPO_EVAL_MIN,
    'csv': 'm4_eval_conteo.csv',
    'limitacion': 'Dataset de peatones en la calle, no de pasajeros dentro de un vehículo.',
    'privacidad': 'Sin reconocimiento facial; no se guardan recortes; visualización con máscara opaca.',
}
json.dump(metricas_json, open('/content/m4_metricas.json', 'w'), indent=2, ensure_ascii=False)
guardar_en_drive([M4_PT, '/content/m4_metricas.json', '/content/m4_eval_conteo.csv'], subcarpeta='metricas/m4')
print(json.dumps(metricas_json['conteo'], indent=2))